# 02 — Data Cleaning & Data Integrity

Notebook này chuyển dữ liệu từ `data/raw/` sang `data/processed/` sau khi kiểm tra và chuẩn hóa có kiểm soát.

**Nguyên tắc:**

1. `data/raw/` chỉ đọc, tuyệt đối không sửa hoặc ghi đè.
2. Không tự động xóa duplicate, dòng `unit_sales` âm, ngày lễ hoặc dòng có missing values.
3. Missing `onpromotion` và `oil.dcoilwtico` được giữ lại trong bước này; quyết định imputing cần có căn cứ và được ghi rõ ở bước feature engineering.
4. Dữ liệu lớn được đọc theo chunks. Nếu có PyArrow, đầu ra là Parquet; nếu không, notebook dùng CSV theo chunks.
5. Mọi phép biến đổi phải giữ nguyên số dòng. Nếu phát hiện lỗi cấu trúc hoặc khóa nghiệp vụ chưa thể xác minh, notebook sẽ dừng thay vì âm thầm sửa dữ liệu.

Thứ tự làm việc: kiểm tra schema → profile chất lượng raw → kiểm tra khóa và tham chiếu → xác nhận các điều kiện an toàn → chuẩn hóa/ghi dữ liệu → kiểm tra đầu ra và lưu báo cáo.


## 0. Configuration & runtime

Chạy notebook từ thư mục project hoặc một thư mục con bên trong project. Notebook tự tìm thư mục gốc có `data/raw/train.csv`.


In [13]:
from pathlib import Path
import json
import math
import os
import platform
import shutil
import time
from collections import defaultdict

import numpy as np
import pandas as pd
from IPython.display import display

try:
    import psutil
except ImportError:
    psutil = None

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
    PARQUET_AVAILABLE = True
except ImportError:
    pa = None
    pq = None
    PARQUET_AVAILABLE = False

CHUNK_SIZE = 100_000
KEY_COLUMNS = ["date", "store_nbr", "item_nbr"]
OUTPUT_FORMAT = "parquet" if PARQUET_AVAILABLE else "csv"


def find_project_root(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    for candidate in candidates:
        if (candidate / "data" / "raw" / "train.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Không tìm thấy data/raw/train.csv. Hãy mở notebook trong repository của project."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_TABLE_DIR = PROJECT_ROOT / "reports" / "table"
REPORT_DIR = PROJECT_ROOT / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Raw directory: {RAW_DIR}")
print(f"Output format: {OUTPUT_FORMAT}")
print(f"Chunk size   : {CHUNK_SIZE:,} rows")
if psutil is not None:
    print(f"Available RAM: {psutil.virtual_memory().available / (1024**3):.2f} GiB")


Project root : /home/phat/Documents/Project/Machine Learning/Grocery-sales-forecast
Raw directory: /home/phat/Documents/Project/Machine Learning/Grocery-sales-forecast/data/raw
Output format: parquet
Chunk size   : 100,000 rows
Available RAM: 1.66 GiB


### Configuration notes

- `CHUNK_SIZE = 100_000` là giá trị khởi đầu an toàn; có thể giảm nếu máy thiếu RAM.
- Parquet được ưu tiên vì tiết kiệm dung lượng và giữ kiểu dữ liệu. Nếu không có PyArrow, notebook tự dùng CSV.
- Đầu ra là các file `*_cleaned.<format>` trong `data/processed/`. Các báo cáo chất lượng nằm trong `reports/` và `reports/table/`.


## 1. Schema validation

Bảng schema ở notebook `01` từng báo `id`, `city`, `cluster`, `state`, `type`, `family`, `class`, `perishable`, `dcoilwtico`, `description`, `locale`, `locale_name`, `transferred`, `transactions` là `extra_columns`. Đó không phải missing columns: các trường này chỉ chưa được khai báo trong tập cột kỳ vọng của cell kiểm tra schema.

Ở đây dùng schema chuẩn đầy đủ của bộ dữ liệu Favorita. `missing_required_columns` là những cột thực sự thiếu; `extra_columns` là cột ngoài schema đã khai báo và cần xem xét, không tự động xóa.


In [14]:
EXPECTED_COLUMNS = {
    "train.csv": {
        "id", "date", "store_nbr", "item_nbr", "unit_sales", "onpromotion"
    },
    "test.csv": {
        "id", "date", "store_nbr", "item_nbr", "onpromotion"
    },
    "stores.csv": {"store_nbr", "city", "state", "type", "cluster"},
    "items.csv": {"item_nbr", "family", "class", "perishable"},
    "oil.csv": {"date", "dcoilwtico"},
    "holidays_events.csv": {
        "date", "type", "locale", "locale_name", "description", "transferred"
    },
    "transactions.csv": {"date", "store_nbr", "transactions"},
}

missing_files = [name for name in EXPECTED_COLUMNS if not (RAW_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(f"Thiếu file trong data/raw/: {missing_files}")

schema_records = []
for filename, expected in EXPECTED_COLUMNS.items():
    actual = set(pd.read_csv(RAW_DIR / filename, nrows=0).columns)
    missing = sorted(expected - actual)
    extra = sorted(actual - expected)
    status = "MISSING_REQUIRED" if missing else ("REVIEW_EXTRA" if extra else "OK")
    schema_records.append({
        "file": filename,
        "status": status,
        "missing_required_columns": missing,
        "extra_columns": extra,
        "actual_column_count": len(actual),
        "expected_column_count": len(expected),
    })

schema_df = pd.DataFrame(schema_records)
display(schema_df)
schema_df.to_csv(REPORT_TABLE_DIR / "02_schema_validation.csv", index=False)

if (schema_df["status"] == "MISSING_REQUIRED").any():
    raise ValueError(
        "Có cột bắt buộc bị thiếu. Dừng trước cleaning; hãy xác minh đúng phiên bản dataset."
    )
if (schema_df["status"] == "REVIEW_EXTRA").any():
    print("CẢNH BÁO: Có extra columns. Chúng sẽ được giữ lại, không tự động loại bỏ.")


,file,status,missing_required_columns,extra_columns,actual_column_count,expected_column_count
0,train.csv,OK,[],[],6,6
1,test.csv,OK,[],[],5,5
2,stores.csv,OK,[],[],5,5
3,items.csv,OK,[],[],4,4
4,oil.csv,OK,[],[],2,2
5,holidays_events.csv,OK,[],[],6,6
6,transactions.csv,OK,[],[],3,3


### Schema — interpretation

Nếu `status = OK`, tên cột đã khớp schema chuẩn. Nếu `REVIEW_EXTRA`, cần xác định cột bổ sung có chủ đích hay không; notebook không xóa chúng. Nếu `MISSING_REQUIRED`, dừng để tránh xử lý nhầm dataset hoặc nhầm phiên bản.


## 2. Cleaning policy — quyết định trước khi sửa dữ liệu

| Vấn đề | Chính sách ở notebook 02 | Lý do |
|---|---|---|
| `unit_sales < 0` | Giữ nguyên, thống kê riêng | Có thể phản ánh trả hàng/điều chỉnh; không mặc định là lỗi |
| `unit_sales == 0` | Giữ nguyên | Không tự xem là invalid |
| `onpromotion` missing | Chuẩn hóa kiểu Boolean nullable, giữ missing | Không tự suy đoán missing là `False` |
| `oil.dcoilwtico` missing | Giữ missing | Imputation theo thời gian cần quyết định riêng |
| Ngày không xuất hiện trong train | Không tự thêm dòng | Không đủ căn cứ coi doanh số ngày đó bằng 0 |
| Duplicate business key | Phát hiện; không tự xóa | Cần hiểu nguyên nhân trước khi deduplicate |
| Whitespace ở cột text của bảng nhỏ | Trim đầu/cuối | Chuẩn hóa biểu diễn, không đổi nội dung ngữ nghĩa |
| Ngày hợp lệ | Parse thành datetime | Chuẩn hóa kiểu dữ liệu |

Bước này ưu tiên **bảo toàn dữ liệu và chuẩn hóa biểu diễn**. Missing-value imputation, lag/rolling và các phép biến đổi phụ thuộc thời gian sẽ được quyết định ở bước tiếp theo.


## 3. Load small reference tables

Các bảng `stores`, `items`, `oil`, `holidays_events`, `transactions` nhỏ hơn `train.csv` nhiều nên có thể nạp toàn bộ sau khi xác nhận kích thước. Các giá trị này dùng cho referential integrity và validation khóa.


In [15]:
SMALL_FILES = [
    "stores.csv", "items.csv", "oil.csv", "holidays_events.csv", "transactions.csv"
]
small_raw = {name: pd.read_csv(RAW_DIR / name) for name in SMALL_FILES}

stores_ref = set(pd.to_numeric(small_raw["stores.csv"]["store_nbr"], errors="coerce").dropna().astype("int64"))
items_ref = set(pd.to_numeric(small_raw["items.csv"]["item_nbr"], errors="coerce").dropna().astype("int64"))

print(f"Unique store keys in stores.csv: {len(stores_ref):,}")
print(f"Unique item keys in items.csv : {len(items_ref):,}")


Unique store keys in stores.csv: 54
Unique item keys in items.csv : 4,100


## 4. Raw data-quality scan — chunked

Cell này kết hợp kiểm tra missingness, kiểu/giá trị quan trọng, khóa tham chiếu và duplicate business key trong một lượt đọc các file lớn. Không giữ toàn bộ `train.csv` trong RAM.

Để chứng minh tính duy nhất của `(date, store_nbr, item_nbr)` chỉ bằng chunk-level duplicate check, cần biết dữ liệu đã được sắp xếp theo khóa hay chưa. Notebook kiểm tra tính tăng dần toàn cục và so sánh khóa ở ranh giới chunk. Nếu dataset không được sắp xếp, kết quả duplicate không được coi là chứng minh đầy đủ.


In [16]:
PROMOTION_MAP = {
    "true": True, "false": False,
    "1": True, "0": False,
    "yes": True, "no": False,
    "y": True, "n": False,
}


def parse_nullable_boolean(series: pd.Series):
    # Return nullable Boolean values and an unknown-token mask.
    text = series.astype("string").str.strip().str.lower()
    text = text.mask(text.isin(["", "na", "nan", "null", "none"]))
    parsed = text.map(PROMOTION_MAP).astype("boolean")
    unknown = text.notna() & ~text.isin(PROMOTION_MAP.keys())
    return parsed, unknown


def numeric_validation(series: pd.Series):
    numeric = pd.to_numeric(series, errors="coerce")
    nonnumeric = int((series.notna() & numeric.isna()).sum())
    noninteger = int((numeric.notna() & ((numeric % 1) != 0)).sum())
    return numeric, nonnumeric, noninteger


def init_profile(filename, columns):
    return {
        "file": filename,
        "rows": 0,
        "missing_by_column": {column: 0 for column in columns},
        "invalid_date_rows": 0,
        "invalid_id_rows": 0,
        "invalid_store_nbr_rows": 0,
        "invalid_item_nbr_rows": 0,
        "nonpositive_store_nbr_rows": 0,
        "nonpositive_item_nbr_rows": 0,
        "unit_sales_nonnumeric_rows": 0,
        "unit_sales_negative_rows": 0,
        "unit_sales_zero_rows": 0,
        "onpromotion_unrecognized_rows": 0,
        "store_reference_mismatch_rows": 0,
        "item_reference_mismatch_rows": 0,
        "business_key_duplicate_rows_within_chunks": 0,
        "business_key_duplicate_rows_at_chunk_boundaries": 0,
        "business_key_order_is_monotonic": True,
    }


def profile_large_csv(filename):
    path = RAW_DIR / filename
    columns = pd.read_csv(path, nrows=0).columns.tolist()
    profile = init_profile(filename, columns)
    previous_last_key = None

    for chunk in pd.read_csv(path, chunksize=CHUNK_SIZE, low_memory=False):
        profile["rows"] += len(chunk)
        missing_counts = chunk.isna().sum()
        for column, count in missing_counts.items():
            profile["missing_by_column"][column] += int(count)

        # Date validity
        dates = pd.to_datetime(chunk["date"], errors="coerce")
        profile["invalid_date_rows"] += int(dates.isna().sum())

        # Primary identifiers and foreign keys
        if "id" in chunk.columns:
            id_numeric, id_nonnumeric, id_noninteger = numeric_validation(chunk["id"])
            profile["invalid_id_rows"] += int(
                id_nonnumeric + id_noninteger + id_numeric.isna().sum()
            )

        store_numeric, store_nonnumeric, store_noninteger = numeric_validation(chunk["store_nbr"])
        item_numeric, item_nonnumeric, item_noninteger = numeric_validation(chunk["item_nbr"])
        profile["invalid_store_nbr_rows"] += int(
            store_nonnumeric + store_noninteger + store_numeric.isna().sum()
        )
        profile["invalid_item_nbr_rows"] += int(
            item_nonnumeric + item_noninteger + item_numeric.isna().sum()
        )
        profile["nonpositive_store_nbr_rows"] += int((store_numeric <= 0).sum())
        profile["nonpositive_item_nbr_rows"] += int((item_numeric <= 0).sum())
        profile["store_reference_mismatch_rows"] += int(
            (~store_numeric.isin(stores_ref) & store_numeric.notna()).sum()
        )
        profile["item_reference_mismatch_rows"] += int(
            (~item_numeric.isin(items_ref) & item_numeric.notna()).sum()
        )

        if "unit_sales" in chunk.columns:
            sales_numeric, sales_nonnumeric, _ = numeric_validation(chunk["unit_sales"])
            profile["unit_sales_nonnumeric_rows"] += int(sales_nonnumeric)
            profile["unit_sales_negative_rows"] += int((sales_numeric < 0).sum())
            profile["unit_sales_zero_rows"] += int((sales_numeric == 0).sum())

        if "onpromotion" in chunk.columns:
            _, unknown_promotion = parse_nullable_boolean(chunk["onpromotion"])
            profile["onpromotion_unrecognized_rows"] += int(unknown_promotion.sum())

        # Candidate business key: (date, store_nbr, item_nbr)
        keys = pd.DataFrame({
            "date": dates,
            "store_nbr": store_numeric,
            "item_nbr": item_numeric,
        })
        profile["business_key_duplicate_rows_within_chunks"] += int(
            keys.duplicated(subset=KEY_COLUMNS).sum()
        )

        if keys[KEY_COLUMNS].isna().any().any():
            profile["business_key_order_is_monotonic"] = False
            previous_last_key = None
        elif len(keys):
            multi_index = pd.MultiIndex.from_frame(keys[KEY_COLUMNS])
            if not multi_index.is_monotonic_increasing:
                profile["business_key_order_is_monotonic"] = False
            first_key = tuple(multi_index[0])
            last_key = tuple(multi_index[-1])
            if previous_last_key is not None:
                if first_key < previous_last_key:
                    profile["business_key_order_is_monotonic"] = False
                if first_key == previous_last_key:
                    profile["business_key_duplicate_rows_at_chunk_boundaries"] += 1
            previous_last_key = last_key

    local_dups = profile["business_key_duplicate_rows_within_chunks"]
    boundary_dups = profile["business_key_duplicate_rows_at_chunk_boundaries"]
    if profile["business_key_order_is_monotonic"]:
        profile["business_key_duplicate_rows_total_if_sorted"] = local_dups + boundary_dups
    else:
        profile["business_key_duplicate_rows_total_if_sorted"] = None
    return profile


large_profiles = {}
for filename in ["train.csv", "test.csv"]:
    start = time.time()
    print(f"Scanning {filename} ...")
    large_profiles[filename] = profile_large_csv(filename)
    print(f"Finished {filename} in {(time.time() - start) / 60:.1f} min")


Scanning train.csv ...
Finished train.csv in 3.0 min
Scanning test.csv ...
Finished test.csv in 0.1 min


### Raw scan — interpretation

- Missing values are counted before any cleaning and are not confused with invalid values.
- Negative `unit_sales` is a finding, not an automatic deletion rule.
- Promotion tokens outside known true/false representations are flagged; null promotions are allowed and retained as unknown.
- If `business_key_order_is_monotonic` is `True`, chunk-local duplicate checks plus boundary checks cover duplicates across the ordered stream. If it is `False`, global uniqueness is **not proven** and the validation gate below will stop before writing cleaned outputs.


In [17]:
quality_summary_records = []
missingness_records = []
for filename, profile in large_profiles.items():
    row = {k: v for k, v in profile.items() if k != "missing_by_column"}
    quality_summary_records.append(row)
    for column, missing_count in profile["missing_by_column"].items():
        missingness_records.append({
            "file": filename,
            "column": column,
            "missing_count": missing_count,
            "missing_pct": (missing_count / profile["rows"] * 100) if profile["rows"] else np.nan,
        })

quality_summary_df = pd.DataFrame(quality_summary_records)
missingness_df = pd.DataFrame(missingness_records)

display(quality_summary_df.T)
display(missingness_df.sort_values(["file", "missing_count"], ascending=[True, False]))
quality_summary_df.to_csv(REPORT_TABLE_DIR / "02_raw_quality_summary.csv", index=False)
missingness_df.to_csv(REPORT_TABLE_DIR / "02_raw_missingness.csv", index=False)


,0,1
file,train.csv,test.csv
rows,125497040,3370464
invalid_date_rows,0,0
invalid_id_rows,0,0
invalid_store_nbr_rows,0,0
invalid_item_nbr_rows,0,0
nonpositive_store_nbr_rows,0,0
nonpositive_item_nbr_rows,0,0
unit_sales_nonnumeric_rows,0,0
unit_sales_negative_rows,7795,0


,file,column,missing_count,missing_pct
6,test.csv,id,0,0.000000
7,test.csv,date,0,0.000000
8,test.csv,store_nbr,0,0.000000
9,test.csv,item_nbr,0,0.000000
10,test.csv,onpromotion,0,0.000000
5,train.csv,onpromotion,21657651,17.257499
0,train.csv,id,0,0.000000
1,train.csv,date,0,0.000000
2,train.csv,store_nbr,0,0.000000
3,train.csv,item_nbr,0,0.000000


## 5. Reference-table integrity & duplicate checks

Một số bảng có khóa duy nhất tự nhiên; riêng `holidays_events.csv` có thể có nhiều event cùng ngày, vì vậy không được coi `date` là khóa duy nhất của bảng này.


In [18]:
UNIQUE_REFERENCE_KEYS = {
    "stores.csv": ["store_nbr"],
    "items.csv": ["item_nbr"],
    "oil.csv": ["date"],
    "transactions.csv": ["date", "store_nbr"],
}

reference_duplicate_records = []
small_table_quality_records = []

for filename, df in small_raw.items():
    missing = df.isna().sum()
    for column, count in missing.items():
        small_table_quality_records.append({
            "file": filename,
            "column": column,
            "rows": len(df),
            "missing_count": int(count),
            "missing_pct": float(count / len(df) * 100) if len(df) else np.nan,
        })

    if filename in UNIQUE_REFERENCE_KEYS:
        key_columns = UNIQUE_REFERENCE_KEYS[filename]
        duplicate_rows = int(df.duplicated(subset=key_columns).sum())
        reference_duplicate_records.append({
            "file": filename,
            "key_columns": key_columns,
            "duplicate_rows": duplicate_rows,
        })

small_missingness_df = pd.DataFrame(small_table_quality_records)
reference_duplicates_df = pd.DataFrame(reference_duplicate_records)

display(small_missingness_df.sort_values(["file", "missing_count"], ascending=[True, False]))
display(reference_duplicates_df)
small_missingness_df.to_csv(REPORT_TABLE_DIR / "02_small_table_missingness.csv", index=False)
reference_duplicates_df.to_csv(REPORT_TABLE_DIR / "02_reference_key_duplicates.csv", index=False)


,file,column,rows,missing_count,missing_pct
11,holidays_events.csv,date,350,0,0.000000
12,holidays_events.csv,type,350,0,0.000000
13,holidays_events.csv,locale,350,0,0.000000
14,holidays_events.csv,locale_name,350,0,0.000000
15,holidays_events.csv,description,350,0,0.000000
16,holidays_events.csv,transferred,350,0,0.000000
5,items.csv,item_nbr,4100,0,0.000000
6,items.csv,family,4100,0,0.000000
7,items.csv,class,4100,0,0.000000
8,items.csv,perishable,4100,0,0.000000


,file,key_columns,duplicate_rows
0,stores.csv,[store_nbr],0
1,items.csv,[item_nbr],0
2,oil.csv,[date],0
3,transactions.csv,"[date, store_nbr]",0


### Reference-table decisions

- `stores.store_nbr`, `items.item_nbr`, `oil.date` và `(transactions.date, transactions.store_nbr)` được kỳ vọng duy nhất.
- Không áp đặt uniqueness cho `holidays_events.date`, vì một ngày có thể có nhiều event.
- Missing oil prices được giữ nguyên; không forward-fill/interpolate trong notebook cleaning.
- Nếu phát hiện duplicate trong bảng cần unique key, cần điều tra và quyết định riêng thay vì xóa dòng tự động.


## 6. Validation gate — có đủ điều kiện làm sạch không?

Trước khi ghi đầu ra, xác nhận không có vấn đề cấu trúc không thể xử lý an toàn bằng chuẩn hóa. Notebook không tự drop hoặc impute để biến validation fail thành pass.


In [19]:
blocking_issues = []

# Required schema
if (schema_df["status"] == "MISSING_REQUIRED").any():
    blocking_issues.append("Required schema columns are missing.")

# Large-table structural validity and referential integrity
for filename, profile in large_profiles.items():
    for field in ["invalid_date_rows", "invalid_store_nbr_rows", "invalid_item_nbr_rows",
                  "nonpositive_store_nbr_rows", "nonpositive_item_nbr_rows",
                  "store_reference_mismatch_rows", "item_reference_mismatch_rows",
                  "onpromotion_unrecognized_rows"]:
        value = int(profile.get(field, 0) or 0)
        if value > 0:
            blocking_issues.append(f"{filename}: {field}={value}")

    if int(profile.get("invalid_id_rows", 0) or 0) > 0:
        blocking_issues.append(f"{filename}: invalid_id_rows={profile['invalid_id_rows']}")

    if filename == "train.csv":
        if int(profile.get("unit_sales_nonnumeric_rows", 0) or 0) > 0:
            blocking_issues.append(
                f"train.csv: unit_sales_nonnumeric_rows={profile['unit_sales_nonnumeric_rows']}"
            )
        if int(profile["missing_by_column"].get("unit_sales", 0)) > 0:
            blocking_issues.append(
                f"train.csv: unit_sales missing={profile['missing_by_column']['unit_sales']}"
            )

    if not profile["business_key_order_is_monotonic"]:
        blocking_issues.append(
            f"{filename}: business key stream is not monotonic; chunk-only duplicate check cannot prove global uniqueness."
        )
    elif int(profile.get("business_key_duplicate_rows_total_if_sorted", 0) or 0) > 0:
        blocking_issues.append(
            f"{filename}: duplicate business-key rows={profile['business_key_duplicate_rows_total_if_sorted']}"
        )

if not reference_duplicates_df.empty:
    duplicates_found = reference_duplicates_df[reference_duplicates_df["duplicate_rows"] > 0]
    for _, row in duplicates_found.iterrows():
        blocking_issues.append(
            f"{row['file']}: duplicate rows for key {row['key_columns']}={row['duplicate_rows']}"
        )

if blocking_issues:
    display(pd.DataFrame({"blocking_issue": blocking_issues}))
    raise RuntimeError(
        "Validation gate failed. Không ghi dữ liệu cleaned. Xử lý từng vấn đề có căn cứ rồi chạy lại notebook."
    )
else:
    print("VALIDATION GATE PASSED: có thể chuyển sang chuẩn hóa và ghi dữ liệu.")


VALIDATION GATE PASSED: có thể chuyển sang chuẩn hóa và ghi dữ liệu.


### Gate interpretation

Đi qua gate không có nghĩa dữ liệu hoàn toàn không còn missing values. Nó nghĩa là các điều kiện cấu trúc cần thiết để chuẩn hóa an toàn đã đạt: schema bắt buộc có mặt, ngày và khóa hợp lệ, quan hệ store/item hợp lệ, token promotion hiểu được, khóa nghiệp vụ có thể kiểm tra toàn cục theo thứ tự dữ liệu và không phát hiện duplicate theo khóa.

Các missing values có chủ đích/đang cần quyết định (`onpromotion`, oil price) được giữ lại và đưa vào báo cáo, không bị che đi bằng imputation mặc định.


## 7. Transform helpers — normalize without dropping rows

Các hàm dưới đây không lọc dòng. Chúng chỉ parse ngày, chuẩn hóa kiểu khóa và chuyển `onpromotion` về Boolean nullable. Bản ghi có promotion missing vẫn còn missing.


In [20]:
def normalize_large_chunk(chunk: pd.DataFrame, filename: str) -> pd.DataFrame:
    out = chunk.copy()

    out["date"] = pd.to_datetime(out["date"], errors="raise")
    out["store_nbr"] = pd.to_numeric(out["store_nbr"], errors="raise").astype("int32")
    out["item_nbr"] = pd.to_numeric(out["item_nbr"], errors="raise").astype("int32")

    if "id" in out.columns:
        out["id"] = pd.to_numeric(out["id"], errors="raise").astype("int64")

    if "unit_sales" in out.columns:
        # Preserve values, including negative unit sales; no rounding or row filtering.
        out["unit_sales"] = pd.to_numeric(out["unit_sales"], errors="raise").astype("float64")

    promotion, unknown = parse_nullable_boolean(out["onpromotion"])
    if int(unknown.sum()) > 0:
        raise ValueError(f"{filename}: found unrecognized onpromotion tokens during transform.")
    out["onpromotion"] = promotion

    return out


def normalize_small_table(filename: str, frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()

    # Trim whitespace for string-like columns without changing their semantic values.
    for column in out.select_dtypes(include=["object", "string"]).columns:
        out[column] = out[column].astype("string").str.strip()

    if "date" in out.columns:
        out["date"] = pd.to_datetime(out["date"], errors="raise")

    if "store_nbr" in out.columns:
        out["store_nbr"] = pd.to_numeric(out["store_nbr"], errors="raise").astype("int32")
    if "item_nbr" in out.columns:
        out["item_nbr"] = pd.to_numeric(out["item_nbr"], errors="raise").astype("int32")

    if filename == "oil.csv":
        out["dcoilwtico"] = pd.to_numeric(out["dcoilwtico"], errors="raise")
    if filename == "transactions.csv":
        out["transactions"] = pd.to_numeric(out["transactions"], errors="raise")

    return out


def write_frame(frame: pd.DataFrame, output_path: Path):
    # Write one small table using the selected output format.
    if OUTPUT_FORMAT == "parquet":
        frame.to_parquet(output_path, index=False, compression="snappy")
    else:
        frame.to_csv(output_path, index=False)


## 8. Write cleaned train/test in chunks

Mỗi file được ghi ra file tạm trước. Chỉ sau khi ghi hết các chunks thành công mới thay thế tên đầu ra chính. Như vậy nếu lỗi giữa chừng, file đầu ra cũ không bị thay bằng một file viết dở.


In [21]:
def clean_large_file(filename: str, output_stem: str):
    input_path = RAW_DIR / filename
    output_path = PROCESSED_DIR / f"{output_stem}_cleaned.{OUTPUT_FORMAT}"
    temp_path = PROCESSED_DIR / f".{output_stem}_cleaned.tmp.{OUTPUT_FORMAT}"

    if temp_path.exists():
        temp_path.unlink()

    row_count = 0
    output_missing_promotion = 0
    output_negative_target = 0
    parquet_writer = None
    first_csv_chunk = True
    start = time.time()

    try:
        for chunk in pd.read_csv(input_path, chunksize=CHUNK_SIZE, low_memory=False):
            cleaned = normalize_large_chunk(chunk, filename)
            row_count += len(cleaned)
            output_missing_promotion += int(cleaned["onpromotion"].isna().sum())
            if "unit_sales" in cleaned.columns:
                output_negative_target += int((cleaned["unit_sales"] < 0).sum())

            if OUTPUT_FORMAT == "parquet":
                table = pa.Table.from_pandas(cleaned, preserve_index=False)
                if parquet_writer is None:
                    parquet_writer = pq.ParquetWriter(
                        temp_path, table.schema, compression="snappy"
                    )
                else:
                    table = pa.Table.from_pandas(
                        cleaned, schema=parquet_writer.schema, preserve_index=False
                    )
                parquet_writer.write_table(table, row_group_size=CHUNK_SIZE)
            else:
                cleaned.to_csv(
                    temp_path,
                    mode="w" if first_csv_chunk else "a",
                    header=first_csv_chunk,
                    index=False,
                )
                first_csv_chunk = False

        if parquet_writer is not None:
            parquet_writer.close()
            parquet_writer = None

        if not temp_path.exists():
            raise RuntimeError(f"No output was produced for {filename}.")

        os.replace(temp_path, output_path)
    except Exception:
        if parquet_writer is not None:
            parquet_writer.close()
        if temp_path.exists():
            temp_path.unlink()
        raise

    result = {
        "input_file": filename,
        "output_file": str(output_path.relative_to(PROJECT_ROOT)),
        "rows_written": row_count,
        "missing_onpromotion_preserved": output_missing_promotion,
        "negative_unit_sales_preserved": output_negative_target if filename == "train.csv" else None,
        "output_size_mb": output_path.stat().st_size / (1024**2),
        "elapsed_minutes": (time.time() - start) / 60,
    }
    return result


large_write_results = []
for filename, output_stem in [("train.csv", "train"), ("test.csv", "test")]:
    print(f"Cleaning/normalizing {filename} ...")
    result = clean_large_file(filename, output_stem)
    large_write_results.append(result)
    display(pd.DataFrame([result]))


Cleaning/normalizing train.csv ...


,input_file,output_file,rows_written,missing_onpromotion_preserved,negative_unit_sales_preserved,output_size_mb,elapsed_minutes
0,train.csv,data/processed/train_cleaned.parquet,125497040,21657651,7795,1102.275264,3.166396


Cleaning/normalizing test.csv ...


,input_file,output_file,rows_written,missing_onpromotion_preserved,negative_unit_sales_preserved,output_size_mb,elapsed_minutes
0,test.csv,data/processed/test_cleaned.parquet,3370464,0,None,25.034229,0.071642


### Large-table transformation — interpretation

- Số dòng được giữ nguyên; không có `dropna`, filtering hoặc deduplication.
- `unit_sales` âm vẫn có trong đầu ra và được đếm để kiểm tra việc bảo toàn.
- `onpromotion` được chuẩn hóa thành kiểu Boolean nullable khi dùng Parquet; ở CSV, giá trị được ghi thành `True`/`False` và ô trống cho missing.
- Output chỉ chứa dữ liệu dẫn xuất trong `data/processed/`; raw files không bị thay đổi.


## 9. Clean small reference tables

Các bảng nhỏ được chuẩn hóa ngày, trim whitespace của các cột text và ép kiểu cho các khóa số. Không loại duplicate, không fill missing oil price và không tự gộp nhiều holiday cùng ngày.


In [22]:
small_write_results = []
for filename, frame in small_raw.items():
    cleaned_small = normalize_small_table(filename, frame)
    output_stem = filename.removesuffix(".csv")
    output_path = PROCESSED_DIR / f"{output_stem}_cleaned.{OUTPUT_FORMAT}"
    write_frame(cleaned_small, output_path)
    small_write_results.append({
        "input_file": filename,
        "output_file": str(output_path.relative_to(PROJECT_ROOT)),
        "rows_input": len(frame),
        "rows_written": len(cleaned_small),
        "output_size_mb": output_path.stat().st_size / (1024**2),
    })

display(pd.DataFrame(small_write_results))


,input_file,output_file,rows_input,rows_written,output_size_mb
0,stores.csv,data/processed/stores_cleaned.parquet,54,54,0.003917
1,items.csv,data/processed/items_cleaned.parquet,4100,4100,0.033337
2,oil.csv,data/processed/oil_cleaned.parquet,1218,1218,0.017352
3,holidays_events.csv,data/processed/holidays_events_cleaned.parquet,350,350,0.008315
4,transactions.csv,data/processed/transactions_cleaned.parquet,83488,83488,0.170516


## 10. Output validation

Kiểm tra nhanh đầu ra sau khi ghi:

1. Các file đầu ra tồn tại và mở được.
2. Số dòng đầu ra bằng số dòng đầu vào.
3. Schema đầu ra vẫn giữ các cột gốc (không có cột nào bị âm thầm bỏ).
4. Missing promotion và giá trị target âm đã thống kê khi ghi.

Với Parquet, số dòng được đọc từ metadata; với CSV, đếm dòng theo chunks.


In [23]:
def output_path_for(input_filename: str) -> Path:
    return PROCESSED_DIR / f"{input_filename.removesuffix('.csv')}_cleaned.{OUTPUT_FORMAT}"


def count_output_rows(path: Path) -> int:
    if OUTPUT_FORMAT == "parquet":
        return int(pq.ParquetFile(path).metadata.num_rows)
    count = 0
    for chunk in pd.read_csv(path, chunksize=CHUNK_SIZE, low_memory=False):
        count += len(chunk)
    return count


expected_row_counts = {
    filename: large_profiles[filename]["rows"] for filename in ["train.csv", "test.csv"]
}
expected_row_counts.update({filename: len(frame) for filename, frame in small_raw.items()})

output_validation_records = []
all_input_files = ["train.csv", "test.csv", *SMALL_FILES]
for filename in all_input_files:
    output_path = output_path_for(filename)
    if not output_path.exists():
        raise FileNotFoundError(f"Missing output: {output_path}")

    output_rows = count_output_rows(output_path)
    input_rows = int(expected_row_counts[filename])
    if OUTPUT_FORMAT == "parquet":
        parquet_file = pq.ParquetFile(output_path)
        first_batch = next(parquet_file.iter_batches(batch_size=5), None)
        sample = (
            first_batch.to_pandas()
            if first_batch is not None
            else pd.DataFrame(columns=parquet_file.schema_arrow.names)
        )
    else:
        sample = pd.read_csv(output_path, nrows=5)

    missing_columns = sorted(set(EXPECTED_COLUMNS[filename]) - set(sample.columns))
    status = "OK" if (output_rows == input_rows and not missing_columns) else "CHECK_FAILED"
    output_validation_records.append({
        "input_file": filename,
        "output_file": str(output_path.relative_to(PROJECT_ROOT)),
        "input_rows": input_rows,
        "output_rows": output_rows,
        "row_count_match": output_rows == input_rows,
        "missing_expected_columns": missing_columns,
        "status": status,
        "output_size_mb": output_path.stat().st_size / (1024**2),
    })

output_validation_df = pd.DataFrame(output_validation_records)
display(output_validation_df)
output_validation_df.to_csv(REPORT_TABLE_DIR / "02_output_validation.csv", index=False)

if (output_validation_df["status"] != "OK").any():
    raise RuntimeError("Output validation failed. Review reports/table/02_output_validation.csv.")


,input_file,output_file,input_rows,output_rows,row_count_match,missing_expected_columns,status,output_size_mb
0,train.csv,data/processed/train_cleaned.parquet,125497040,125497040,True,[],OK,1102.275264
1,test.csv,data/processed/test_cleaned.parquet,3370464,3370464,True,[],OK,25.034229
2,stores.csv,data/processed/stores_cleaned.parquet,54,54,True,[],OK,0.003917
3,items.csv,data/processed/items_cleaned.parquet,4100,4100,True,[],OK,0.033337
4,oil.csv,data/processed/oil_cleaned.parquet,1218,1218,True,[],OK,0.017352
5,holidays_events.csv,data/processed/holidays_events_cleaned.parquet,350,350,True,[],OK,0.008315
6,transactions.csv,data/processed/transactions_cleaned.parquet,83488,83488,True,[],OK,0.170516


### Output validation — interpretation

`row_count_match = True` cho mọi file chứng minh rằng bước cleaning này không loại hoặc thêm record. Đây là điều có chủ đích ở bước đầu: phát hiện và chuẩn hóa trước, không tự ý thay đổi dữ liệu chỉ để làm bảng trông sạch hơn.


## 11. Save consolidated cleaning report

Báo cáo JSON lưu lại policy, profile raw, kết quả ghi và kiểm tra output để có thể tra cứu mà không phải suy luận từ từng cell.


In [24]:
def make_json_safe(value):
    if isinstance(value, dict):
        return {str(k): make_json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [make_json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if pd.isna(value) if not isinstance(value, (dict, list, tuple)) else False:
        return None
    return value


cleaning_report = {
    "notebook": "02_data_cleaning.ipynb",
    "project_root": str(PROJECT_ROOT),
    "output_format": OUTPUT_FORMAT,
    "chunk_size": CHUNK_SIZE,
    "policies": {
        "raw_files_modified": False,
        "rows_dropped": False,
        "duplicates_deleted": False,
        "negative_unit_sales_deleted": False,
        "onpromotion_missing_imputed": False,
        "oil_missing_imputed": False,
        "date_gaps_filled": False,
        "date_normalization": "datetime",
        "onpromotion_normalization": "nullable boolean; missing preserved",
    },
    "schema_validation": schema_records,
    "raw_large_table_profiles": large_profiles,
    "reference_key_duplicates": reference_duplicate_records,
    "large_table_write_results": large_write_results,
    "small_table_write_results": small_write_results,
    "output_validation": output_validation_records,
}

report_path = REPORT_DIR / "02_cleaning_report.json"
with report_path.open("w", encoding="utf-8") as file:
    json.dump(make_json_safe(cleaning_report), file, ensure_ascii=False, indent=2)

print(f"Cleaning report saved: {report_path.relative_to(PROJECT_ROOT)}")


Cleaning report saved: reports/02_cleaning_report.json


## 12. Final checklist & handoff to `03_feature_engineering`

Sau khi chạy hết notebook, xác nhận:

- [ ] Schema bắt buộc đã xác nhận; cột extra (nếu có) đã được xem xét.
- [ ] Không có lỗi ngày/khóa/kiểu dữ liệu không giải thích được.
- [ ] Kiểm tra duplicate business key đã có kết luận rõ ràng.
- [ ] Số dòng raw và processed khớp nhau ở tất cả file.
- [ ] Raw files vẫn nguyên vẹn.
- [ ] Missing `onpromotion` và `oil.dcoilwtico` được giữ lại, có báo cáo.
- [ ] Negative `unit_sales` không bị xóa tự động.
- [ ] Các bảng báo cáo đã lưu trong `reports/table/` và `reports/02_cleaning_report.json`.

### Quyết định chuyển tiếp

`02_data_cleaning` chỉ chuẩn hóa và xác minh dữ liệu; không thực hiện split ngẫu nhiên, không tạo lag/rolling và không impute theo cách có thể gây leakage. Sang `03_feature_engineering`, P sẽ quyết định rõ chính sách imputing theo thời gian, tạo calendar features, promotion/oil features và chiến lược feature cho forecasting.

**Đầu ra chính:** `data/processed/*_cleaned.<parquet|csv>`.
